In [ ]:
Name=Harshit Singh
Roll no.= 2409301043
Course = BCA

In [ ]:
AI PROMPT :-
A data visualization dashboard using Jupyter Notebooks to analyse global COVID-19 data using interactive visualisation. 
The dashboard allows users to create interactive data visualisation using Plotly; 
static data visualisation with Matplotlib; and manipulate the data using Pandas, Seaborn and IpyWidgets.

In addition, the dashboard should include the following:

- A drop-down list of cities that can be compared

- A comparison between multiple cities at once

- A date range filter (slider or calendar)

- A line chart displaying the trends of COVID-19 cases over time

- A bar chart comparing multiple cities

- A pie chart displaying how COVID-19 relates to the number of active cases, number of recovered cases and number of deaths.

- Key Performance Indicators including: Total Confirmed Cases, Total Active Cases, Total Deaths, Total Recovered Cases

- A heatmap displaying correlation (using Seaborn) between confirmed cases, deaths, recovered cases and active cases

- A way to calculate daily growth rates of COVID-19 cases

- Clean code with comments for documentation

- Use of Plotly Express and graphic objects where appropriate

- Responsive and interactive visualisation of the data visualisation

- Appropriate handling of missing values

All the above will complete a full code implementation that can run without errors. 
Please note that it will not include any HTML or CSS within the code and must use only interactive graphs in the final deliverable.

# 🦠 COVID-19 Urban Area Interactive Dashboard
### City-Level Analysis | Time-Series Trends | Multi-City Comparisons
---

## 📦 Cell 1 — Install & Import Libraries

In [2]:
# ── Install required packages (run once) ──────────────────────────────────────
import subprocess, sys
packages = ['plotly', 'ipywidgets', 'openpyxl']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages ready.')

✅ All packages ready.


## 📚 Cell 2 — Imports

In [3]:
# ── Core data libraries ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation libraries ───────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# ── Interactive widgets ───────────────────────────────────────────────────────
import ipywidgets as widgets
from ipywidgets import interact, interactive, HBox, VBox, Output, Layout
from IPython.display import display, HTML, clear_output

# ── Notebook settings ─────────────────────────────────────────────────────────
pd.set_option('display.float_format', '{:,.0f}'.format)
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Imports complete.')

✅ Imports complete.


## 🗄️ Cell 3 — Load & Clean Data

In [16]:
# ── Load dataset ──────────────────────────────────────────────────────────────
FILE_PATH = 'Covid_19_Urban_Area_Data.xlsx'   # ← adjust path if needed

raw = pd.read_excel(FILE_PATH, parse_dates=['Reported_Date'])

# ── Rename columns for convenience ───────────────────────────────────────────
df = raw.rename(columns={
    'City_Name'                  : 'city',
    'City_Class'                 : 'city_class',
    'Country_Name'               : 'country',
    'Sub_National_Region_Name'   : 'region',
    'Reported_Date'              : 'date',
    'Cases'                      : 'confirmed',
    'Deaths'                     : 'deaths',
    'Recovered'                  : 'recovered',
    'Case_Rate_per_100000'       : 'case_rate',
    'Case_Fatality_Rate'         : 'cfr',
    'Population_Estimate_2020_2021': 'population',
    'Latitude'                   : 'lat',
    'Longitude'                  : 'lon',
}).copy()

# ── Drop empty helper columns ─────────────────────────────────────────────────
df.drop(columns=[c for c in df.columns if c.startswith('Unnamed')], inplace=True)

# ── Coerce numerics ───────────────────────────────────────────────────────────
for col in ['confirmed', 'deaths', 'cfr', 'lat', 'lon']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Sort & reset index ────────────────────────────────────────────────────────
df.sort_values(['city', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Derived metric: active cases (best-effort) ────────────────────────────────
df['active'] = (df['confirmed'] - df['deaths'].fillna(0) - df['recovered'].fillna(0)).clip(lower=0)

# ── Derived metric: daily new cases per city ─────────────────────────────────
df['daily_new'] = df.groupby('city')['confirmed'].diff().clip(lower=0)

# ── Derived metric: 7-day rolling growth rate ─────────────────────────────────
df['growth_rate'] = (
    df.groupby('city')['confirmed']
      .transform(lambda s: s.pct_change(periods=7) * 100)
      .clip(lower=-100, upper=500)
)

# ── Summary ───────────────────────────────────────────────────────────────────
CITIES   = sorted(df['city'].dropna().unique().tolist())
COUNTRIES= sorted(df['country'].dropna().unique().tolist())
DATE_MIN = df['date'].min()
DATE_MAX = df['date'].max()

print(f'✅ Dataset loaded:  {len(df):,} rows | {df["city"].nunique():,} cities | {df["country"].nunique()} countries')
print(f'   Date range: {DATE_MIN.date()} → {DATE_MAX.date()}')

✅ Dataset loaded:  309,480 rows | 20,898 cities | 142 countries
   Date range: 2020-05-02 → 2022-02-28


## 🔧 Cell 4 — Helper Functions

In [19]:
# ── Colour palette for multi-city comparisons ─────────────────────────────────
PALETTE = px.colors.qualitative.Bold

def city_data(city_name: str) -> pd.DataFrame:
    """Return time-series rows for a single city, sorted by date."""
    return df[df['city'] == city_name].sort_values('date')

def latest_kpis(city_name: str) -> dict:
    """Return the most-recent row's KPI values for a city."""
    cdf = city_data(city_name)
    if cdf.empty:
        return {k: np.nan for k in ['confirmed','active','recovered','deaths']}
    last = cdf.dropna(subset=['confirmed']).iloc[-1]
    return {
        'confirmed' : int(last['confirmed'])  if pd.notna(last['confirmed'])  else 0,
        'deaths'    : int(last['deaths'])     if pd.notna(last['deaths'])     else 0,
        'recovered' : int(last['recovered'])  if pd.notna(last['recovered'])  else 0,
        'active'    : int(last['active'])     if pd.notna(last['active'])     else 0,
    }

def fmt(n: int) -> str:
    """Format large integers with K/M suffix."""
    if n >= 1_000_000: return f'{n/1_000_000:.2f}M'
    if n >= 1_000:     return f'{n/1_000:.1f}K'
    return str(n)

def kpi_html(kpis: dict, city: str) -> str:
    """Render Bootstrap-style KPI cards as an HTML string."""
    cards = [
        ('🦠 Total Confirmed', kpis['confirmed'], '#4361ee'),
        ('🏥 Active Cases',    kpis['active'],    '#f77f00'),
        ('💚 Recovered',       kpis['recovered'], '#2dc653'),
        ('💀 Deaths',          kpis['deaths'],    '#e63946'),
    ]
    html  = f'<h3 style="font-family:Arial;color:#333;margin-bottom:8px">📍 {city} — Key Indicators</h3>'
    html += '<div style="display:flex;gap:16px;flex-wrap:wrap;">'
    for label, val, color in cards:
        html += f'''
        <div style="background:{color};color:#fff;padding:16px 24px;border-radius:12px;
                    min-width:140px;text-align:center;box-shadow:2px 4px 12px rgba(0,0,0,.2);">
            <div style="font-size:13px;opacity:.85;">{label}</div>
            <div style="font-size:28px;font-weight:700;margin-top:4px">{fmt(val)}</div>
        </div>'''
    html += '</div>'
    return html

print('✅ Helper functions defined.')

✅ Helper functions defined.


---
# 🎛️ Section A — Single-City Deep Dive
### KPI Cards · Line Trends · Pie Chart · Daily Growth Rate

In [20]:
# ── Widget definitions ────────────────────────────────────────────────────────
TOP_CITIES = (
    df.groupby('city')['confirmed']
      .apply(lambda s: pd.to_numeric(s, errors='coerce').max())
      .nlargest(200)
      .index.tolist()
)

w_city = widgets.Combobox(
    options=TOP_CITIES,
    value=TOP_CITIES[0],
    description='🏙️ City:',
    ensure_option=False,
    style={'description_width': '60px'},
    layout=Layout(width='340px')
)

all_dates = pd.date_range(DATE_MIN, DATE_MAX, freq='MS')
w_date_start = widgets.SelectionSlider(
    options=[(d.strftime('%b %Y'), d) for d in all_dates],
    value=all_dates[0],
    description='📅 From:',
    style={'description_width': '50px'},
    layout=Layout(width='440px')
)
w_date_end = widgets.SelectionSlider(
    options=[(d.strftime('%b %Y'), d) for d in all_dates],
    value=all_dates[-1],
    description='📅 To:',
    style={'description_width': '50px'},
    layout=Layout(width='440px')
)
w_metric = widgets.ToggleButtons(
    options=['confirmed','active','recovered','deaths','daily_new','growth_rate'],
    value='confirmed',
    description='📊 Metric:',
    style={'description_width': '65px', 'button_width': '110px'}
)
w_smooth = widgets.Checkbox(value=True, description='7-day rolling avg', indent=False)

out_kpi   = Output()
out_line  = Output()
out_pie   = Output()
out_growth= Output()

# ── Render function ───────────────────────────────────────────────────────────
def render_single_city(city, d_start, d_end, metric, smooth):
    cdf = city_data(city)
    cdf = cdf[(cdf['date'] >= pd.Timestamp(d_start)) & (cdf['date'] <= pd.Timestamp(d_end))]

    # --- KPI cards ---
    with out_kpi:
        clear_output(wait=True)
        if not cdf.empty:
            kpis = latest_kpis(city)
            display(HTML(kpi_html(kpis, city)))

    # --- Line chart ---
    with out_line:
        clear_output(wait=True)
        if cdf.empty:
            print('⚠️  No data for this city / date range.')
            return
        y_col = metric
        y_data = cdf[y_col] if not smooth else cdf[y_col].rolling(7, min_periods=1).mean()
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=cdf['date'], y=y_data,
            mode='lines', name=metric,
            line=dict(width=2.5, color='#4361ee'),
            fill='tozeroy', fillcolor='rgba(67,97,238,.15)'
        ))
        fig.update_layout(
            title=dict(text=f'<b>{city}</b> — {metric.replace("_",' ').title()} over Time',
                       font_size=17),
            xaxis_title='Date', yaxis_title=metric,
            hovermode='x unified',
            template='plotly_white',
            height=380,
            margin=dict(t=55, b=40)
        )
        fig.show()

    # --- Pie chart ---
    with out_pie:
        clear_output(wait=True)
        kpis = latest_kpis(city)
        labels = ['Active','Recovered','Deaths']
        vals   = [kpis['active'], kpis['recovered'], kpis['deaths']]
        if sum(vals) == 0:
            print('⚠️  No distribution data available.')
            return
        fig = go.Figure(go.Pie(
            labels=labels, values=vals,
            hole=.42,
            marker_colors=['#f77f00','#2dc653','#e63946'],
            textinfo='label+percent',
            hovertemplate='<b>%{label}</b><br>%{value:,}<extra></extra>'
        ))
        fig.update_layout(
            title=dict(text=f'<b>{city}</b> — Case Distribution', font_size=16),
            legend=dict(orientation='h', y=-0.1),
            height=360, template='plotly_white', margin=dict(t=55)
        )
        fig.show()

    # --- Daily growth rate bar chart ---
    with out_growth:
        clear_output(wait=True)
        gr = cdf[['date','growth_rate']].dropna()
        if gr.empty:
            print('⚠️  Insufficient data to compute growth rate.')
            return
        colors = ['#e63946' if v > 0 else '#2dc653' for v in gr['growth_rate']]
        fig = go.Figure(go.Bar(
            x=gr['date'], y=gr['growth_rate'],
            marker_color=colors,
            hovertemplate='%{x|%d %b %Y}<br>Growth: %{y:.1f}%<extra></extra>'
        ))
        fig.update_layout(
            title=dict(text=f'<b>{city}</b> — 7-Day Rolling Case Growth Rate (%)', font_size=16),
            xaxis_title='Date', yaxis_title='Growth Rate (%)',
            hovermode='x unified', template='plotly_white',
            height=340, margin=dict(t=55, b=40)
        )
        fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=1)
        fig.show()

# ── Wire widgets → render ─────────────────────────────────────────────────────
def on_change(*_):
    render_single_city(
        w_city.value, w_date_start.value, w_date_end.value,
        w_metric.value, w_smooth.value
    )

for w in [w_city, w_date_start, w_date_end, w_metric, w_smooth]:
    w.observe(on_change, names='value')

# ── Layout ────────────────────────────────────────────────────────────────────
display(HTML('<hr><h2 style="font-family:Arial">🔍 Single-City Deep Dive</h2>'))
display(VBox([
    HBox([w_city, w_smooth]),
    w_date_start,
    w_date_end,
    w_metric,
    out_kpi,
    HBox([out_line]),
    HBox([out_pie, out_growth]),
]))

# Initial render
on_change()

---
# 🌐 Section B — Multi-City Comparison
### Bar chart · Overlapping line chart · Bubble map

In [21]:
# ── Widgets ───────────────────────────────────────────────────────────────────
w_cities_multi = widgets.SelectMultiple(
    options=TOP_CITIES,
    value=TOP_CITIES[:6],
    description='🏙️ Cities:',
    rows=8,
    style={'description_width': '60px'},
    layout=Layout(width='320px', height='200px')
)
w_comp_metric = widgets.Dropdown(
    options=['confirmed','deaths','recovered','active','daily_new','case_rate'],
    value='confirmed',
    description='📊 Metric:',
    style={'description_width': '65px'},
    layout=Layout(width='240px')
)
w_comp_date = widgets.SelectionSlider(
    options=[(d.strftime('%b %Y'), d) for d in all_dates],
    value=all_dates[-1],
    description='📅 Up to:',
    style={'description_width': '55px'},
    layout=Layout(width='440px')
)

out_bar  = Output()
out_mline= Output()
out_map  = Output()

# ── Render function ───────────────────────────────────────────────────────────
def render_multi(cities, metric, end_date):
    cities = list(cities)
    if not cities:
        return
    end_ts = pd.Timestamp(end_date)
    sub = df[(df['city'].isin(cities)) & (df['date'] <= end_ts)]

    # --- Bar chart: latest snapshot per city ---
    with out_bar:
        clear_output(wait=True)
        latest = (
            sub.dropna(subset=[metric])
               .sort_values('date')
               .groupby('city')
               .last()
               .reset_index()
               .sort_values(metric, ascending=False)
        )
        fig = px.bar(
            latest, x='city', y=metric,
            color='city', color_discrete_sequence=PALETTE,
            text_auto='.3s',
            title=f'<b>City Comparison</b> — {metric.replace("_"," ").title()} (as of {end_ts.date()})'
        )
        fig.update_layout(
            showlegend=False, template='plotly_white',
            height=400, xaxis_title='', yaxis_title=metric,
            margin=dict(t=55, b=60)
        )
        fig.show()

    # --- Overlapping line chart ---
    with out_mline:
        clear_output(wait=True)
        fig = go.Figure()
        for i, city in enumerate(cities):
            cdf = sub[sub['city'] == city].sort_values('date')
            y   = cdf[metric].rolling(7, min_periods=1).mean()
            fig.add_trace(go.Scatter(
                x=cdf['date'], y=y, name=city,
                mode='lines', line=dict(width=2, color=PALETTE[i % len(PALETTE)])
            ))
        fig.update_layout(
            title=f'<b>Trend Comparison</b> — {metric.replace("_"," ").title()} (7-day avg)',
            xaxis_title='Date', yaxis_title=metric,
            hovermode='x unified', template='plotly_white',
            height=400, margin=dict(t=55, b=40),
            legend=dict(orientation='h', y=-0.2)
        )
        fig.show()

    # --- Bubble map ---
    with out_map:
        clear_output(wait=True)
        map_df = (
            sub.dropna(subset=[metric,'lat','lon'])
               .sort_values('date')
               .groupby('city')
               .last()
               .reset_index()
        )
        map_df['lat'] = pd.to_numeric(map_df['lat'], errors='coerce')
        map_df['lon'] = pd.to_numeric(map_df['lon'], errors='coerce')
        map_df = map_df.dropna(subset=['lat','lon',metric])
        if map_df.empty:
            print('⚠️  No geo data available for selected cities.')
            return
        fig = px.scatter_geo(
            map_df, lat='lat', lon='lon',
            size=metric, color='city',
            hover_name='city', hover_data={metric: ':,', 'country': True, 'lat': False, 'lon': False},
            color_discrete_sequence=PALETTE,
            size_max=50,
            title=f'<b>Geographic View</b> — {metric.replace("_"," ").title()}',
            projection='natural earth'
        )
        fig.update_geos(showcoastlines=True, showland=True, landcolor='#f0f0f0',
                        showocean=True, oceancolor='#cfe2ff')
        fig.update_layout(template='plotly_white', height=440,
                          legend=dict(orientation='h', y=-0.1),
                          margin=dict(t=55, b=0, l=0, r=0))
        fig.show()

# ── Wire ─────────────────────────────────────────────────────────────────────
def on_multi_change(*_):
    render_multi(w_cities_multi.value, w_comp_metric.value, w_comp_date.value)

for w in [w_cities_multi, w_comp_metric, w_comp_date]:
    w.observe(on_multi_change, names='value')

# ── Layout ────────────────────────────────────────────────────────────────────
display(HTML('<hr><h2 style="font-family:Arial">🌐 Multi-City Comparison</h2><p style="color:#555">Hold Ctrl/Cmd to select multiple cities.</p>'))
display(VBox([
    HBox([w_cities_multi, VBox([w_comp_metric, w_comp_date])]),
    out_bar,
    out_mline,
    out_map
]))

on_multi_change()

---
# 🔥 Section C — Correlation Heatmap (Seaborn + Matplotlib)

In [22]:
# ── Widgets ───────────────────────────────────────────────────────────────────
w_hmap_cities = widgets.SelectMultiple(
    options=TOP_CITIES,
    value=TOP_CITIES[:10],
    description='🏙️ Cities:',
    rows=6,
    style={'description_width': '60px'},
    layout=Layout(width='320px', height='160px')
)
w_hmap_cols = widgets.SelectMultiple(
    options=['confirmed','deaths','recovered','active','daily_new','growth_rate',
             'case_rate','population'],
    value=['confirmed','deaths','recovered','active','growth_rate'],
    description='📊 Metrics:',
    rows=6,
    style={'description_width': '65px'},
    layout=Layout(width='260px', height='160px')
)
w_hmap_btn = widgets.Button(
    description='🔄 Render Heatmap',
    button_style='info',
    layout=Layout(width='200px', height='36px')
)
out_hmap = Output()

def render_hmap(_=None):
    with out_hmap:
        clear_output(wait=True)
        cities = list(w_hmap_cities.value)
        cols   = list(w_hmap_cols.value)
        if len(cities) < 2 or len(cols) < 2:
            print('⚠️  Select at least 2 cities and 2 metrics.')
            return

        # Latest snapshot per selected city
        sub = (
            df[df['city'].isin(cities)]
              .dropna(subset=['confirmed'])
              .sort_values('date')
              .groupby('city')
              .last()
              .reset_index()
        )
        available_cols = [c for c in cols if c in sub.columns]
        corr = sub[available_cols].apply(pd.to_numeric, errors='coerce').corr()

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # -- Correlation heatmap
        sns.heatmap(
            corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=.5, ax=axes[0],
            annot_kws={'size': 11}
        )
        axes[0].set_title('Correlation Matrix of Selected Metrics', fontsize=14, fontweight='bold', pad=12)

        # -- Horizontal bar: confirmed cases per city
        bar_data = sub[['city','confirmed']].dropna().sort_values('confirmed')
        colors   = sns.color_palette('Blues_r', len(bar_data))
        bars = axes[1].barh(bar_data['city'], bar_data['confirmed'], color=colors)
        axes[1].set_title('Latest Confirmed Cases by City', fontsize=14, fontweight='bold', pad=12)
        axes[1].set_xlabel('Confirmed Cases')
        axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K' if x >= 1000 else str(int(x))))
        for bar in bars:
            w_val = bar.get_width()
            axes[1].text(w_val * 1.01, bar.get_y() + bar.get_height()/2,
                         fmt(int(w_val)), va='center', fontsize=9, color='#333')

        plt.tight_layout()
        plt.show()

w_hmap_btn.on_click(render_hmap)

display(HTML('<hr><h2 style="font-family:Arial">🔥 Correlation Heatmap &amp; Static Comparison</h2>'))
display(VBox([
    HBox([w_hmap_cities, w_hmap_cols, w_hmap_btn]),
    out_hmap
]))
render_hmap()

---
# 📈 Section D — Animated Global Spread (Plotly Animation)

In [23]:
# ── Build a monthly snapshot with geo coords ──────────────────────────────────
anim_df = df.copy()
anim_df['month'] = anim_df['date'].dt.to_period('M').dt.to_timestamp()
anim_df['lat']   = pd.to_numeric(anim_df['lat'], errors='coerce')
anim_df['lon']   = pd.to_numeric(anim_df['lon'], errors='coerce')

monthly = (
    anim_df.dropna(subset=['confirmed','lat','lon'])
           .sort_values('date')
           .groupby(['city','month'])
           .last()
           .reset_index()
)
monthly['month_str'] = monthly['month'].dt.strftime('%b %Y')

# ── Widget ────────────────────────────────────────────────────────────────────
w_anim_metric = widgets.Dropdown(
    options=['confirmed','deaths','case_rate'],
    value='confirmed',
    description='📊 Metric:',
    layout=Layout(width='220px')
)
w_anim_btn = widgets.Button(
    description='▶ Generate Animation',
    button_style='success',
    layout=Layout(width='200px', height='36px')
)
out_anim = Output()

def render_anim(_=None):
    with out_anim:
        clear_output(wait=True)
        m = w_anim_metric.value
        data = monthly[['city','lat','lon','month_str', m, 'country']].dropna()
        data = data[data[m] > 0]
        fig = px.scatter_geo(
            data, lat='lat', lon='lon', size=m, color='country',
            animation_frame='month_str',
            hover_name='city', size_max=40,
            title=f'<b>Global Spread Animation</b> — {m.replace("_"," ").title()}',
            projection='natural earth'
        )
        fig.update_geos(showcoastlines=True, showland=True, landcolor='#f0f0f0',
                        showocean=True, oceancolor='#cfe2ff')
        fig.update_layout(
            height=520, template='plotly_white',
            legend=dict(orientation='h', y=-0.05, font_size=9),
            margin=dict(t=60, b=0, l=0, r=0)
        )
        fig.show()

w_anim_btn.on_click(render_anim)

display(HTML('<hr><h2 style="font-family:Arial">📈 Animated Global Spread</h2><p style="color:#555">⚠️ Large dataset — rendering may take a moment.</p>'))
display(HBox([w_anim_metric, w_anim_btn]))
display(out_anim)

Output()

---
# 🔬 Section E — Statistical Distribution (Seaborn)

In [24]:
# ── Seaborn distribution and pair plots ───────────────────────────────────────
w_dist_cities = widgets.SelectMultiple(
    options=TOP_CITIES[:50],
    value=TOP_CITIES[:5],
    description='🏙️ Cities:',
    rows=6,
    style={'description_width': '60px'},
    layout=Layout(width='300px', height='150px')
)
w_dist_type = widgets.RadioButtons(
    options=['Box Plot','Violin Plot','KDE Distribution','Scatter Matrix'],
    value='Box Plot',
    description='📊 Plot type:',
    style={'description_width': '80px'}
)
w_dist_btn = widgets.Button(
    description='🔄 Render',
    button_style='warning',
    layout=Layout(width='150px', height='36px')
)
out_dist = Output()

def render_dist(_=None):
    with out_dist:
        clear_output(wait=True)
        cities = list(w_dist_cities.value)
        if not cities:
            return
        sub = df[df['city'].isin(cities)][['city','date','confirmed','deaths','active','growth_rate']].dropna(subset=['confirmed'])
        sub['confirmed'] = pd.to_numeric(sub['confirmed'], errors='coerce')
        sub['deaths']    = pd.to_numeric(sub['deaths'],    errors='coerce')

        fig, ax = plt.subplots(figsize=(14, 5))
        ptype = w_dist_type.value

        if ptype == 'Box Plot':
            sns.boxplot(data=sub, x='city', y='confirmed', palette='Set2', ax=ax)
            ax.set_title('Confirmed Cases Distribution per City (Box Plot)', fontsize=14, fontweight='bold')

        elif ptype == 'Violin Plot':
            sns.violinplot(data=sub, x='city', y='confirmed', palette='muted', inner='quartile', ax=ax)
            ax.set_title('Confirmed Cases Distribution per City (Violin Plot)', fontsize=14, fontweight='bold')

        elif ptype == 'KDE Distribution':
            for city in cities:
                cdf = sub[sub['city'] == city]['confirmed'].dropna()
                if len(cdf) > 5:
                    sns.kdeplot(cdf, label=city, fill=True, alpha=0.35, ax=ax)
            ax.set_title('KDE — Confirmed Cases per City', fontsize=14, fontweight='bold')
            ax.legend()

        elif ptype == 'Scatter Matrix':
            plt.close()
            pair_df = sub[['city','confirmed','deaths','growth_rate']].dropna()
            g = sns.pairplot(pair_df, hue='city', plot_kws={'alpha':.5}, height=2.8)
            g.figure.suptitle('Scatter Matrix', y=1.01, fontsize=14, fontweight='bold')
            plt.show()
            return

        ax.set_xlabel('')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K' if x >= 1000 else str(int(x))))
        plt.xticks(rotation=20, ha='right')
        plt.tight_layout()
        plt.show()

w_dist_btn.on_click(render_dist)

display(HTML('<hr><h2 style="font-family:Arial">🔬 Statistical Distribution (Seaborn)</h2>'))
display(VBox([
    HBox([w_dist_cities, w_dist_type, w_dist_btn]),
    out_dist
]))
render_dist()

---
# 📊 Section F — Top-N Ranking Dashboard

In [25]:
# ── Interactive ranking chart ─────────────────────────────────────────────────
w_rank_n = widgets.IntSlider(
    value=15, min=5, max=50, step=5,
    description='Top N:',
    style={'description_width': '55px'},
    layout=Layout(width='340px')
)
w_rank_metric = widgets.Dropdown(
    options=['confirmed','deaths','recovered','active','case_rate'],
    value='confirmed',
    description='📊 Metric:',
    style={'description_width': '65px'},
    layout=Layout(width='230px')
)
w_rank_country = widgets.Dropdown(
    options=['All'] + COUNTRIES,
    value='All',
    description='🌍 Country:',
    style={'description_width': '68px'},
    layout=Layout(width='260px')
)
out_rank = Output()

def render_ranking(n, metric, country):
    with out_rank:
        clear_output(wait=True)
        sub = df.copy()
        if country != 'All':
            sub = sub[sub['country'] == country]
        latest = (
            sub.dropna(subset=[metric])
               .sort_values('date')
               .groupby('city')
               .last()
               .reset_index()
        )
        latest[metric] = pd.to_numeric(latest[metric], errors='coerce')
        top = latest.nlargest(n, metric)[['city','country', metric]].reset_index(drop=True)
        top.index += 1

        fig = px.bar(
            top.sort_values(metric), x=metric, y='city',
            orientation='h',
            color=metric, color_continuous_scale='Blues',
            text_auto='.3s',
            title=f'<b>Top {n} Cities</b> — {metric.replace("_"," ").title()}' + 
                  (f' ({country})' if country != 'All' else ''),
            hover_data={'country': True}
        )
        fig.update_layout(
            template='plotly_white', coloraxis_showscale=False,
            height=max(350, n * 28), yaxis_title='',
            margin=dict(t=55, b=40, l=10)
        )
        fig.show()

def on_rank_change(*_):
    render_ranking(w_rank_n.value, w_rank_metric.value, w_rank_country.value)

for w in [w_rank_n, w_rank_metric, w_rank_country]:
    w.observe(on_rank_change, names='value')

display(HTML('<hr><h2 style="font-family:Arial">📊 Top-N City Ranking</h2>'))
display(VBox([
    HBox([w_rank_n, w_rank_metric, w_rank_country]),
    out_rank
]))
on_rank_change()

---
# 🗓️ Section G — Daily New Cases Heatmap Calendar (Matplotlib)

In [26]:
# ── Calendar-style heatmap of daily new cases ─────────────────────────────────
w_cal_city = widgets.Combobox(
    options=TOP_CITIES,
    value=TOP_CITIES[0],
    description='🏙️ City:',
    style={'description_width': '55px'},
    layout=Layout(width='320px')
)
w_cal_year = widgets.Dropdown(
    options=[2020, 2021, 2022],
    value=2021,
    description='📅 Year:',
    style={'description_width': '50px'},
    layout=Layout(width='160px')
)
out_cal = Output()

def render_calendar(*_):
    with out_cal:
        clear_output(wait=True)
        city = w_cal_city.value
        year = w_cal_year.value
        cdf  = city_data(city)
        cdf  = cdf[cdf['date'].dt.year == year][['date','daily_new']].dropna()
        if cdf.empty:
            print(f'⚠️  No daily-new data for {city} in {year}.')
            return

        cdf['week']    = cdf['date'].dt.isocalendar().week.astype(int)
        cdf['weekday'] = cdf['date'].dt.weekday  # 0=Mon

        pivot = cdf.pivot_table(index='weekday', columns='week', values='daily_new', aggfunc='sum')
        pivot = pivot.reindex(index=range(7))

        fig, ax = plt.subplots(figsize=(16, 4))
        im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
        plt.colorbar(im, ax=ax, label='Daily New Cases')
        ax.set_yticks(range(7))
        ax.set_yticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
        ax.set_xlabel('Week of Year')
        ax.set_title(f'{city} — Daily New Cases Calendar Heatmap ({year})', fontsize=14, fontweight='bold', pad=12)
        plt.tight_layout()
        plt.show()

for w in [w_cal_city, w_cal_year]:
    w.observe(render_calendar, names='value')

display(HTML('<hr><h2 style="font-family:Arial">🗓️ Daily New Cases — Calendar Heatmap</h2>'))
display(HBox([w_cal_city, w_cal_year]))
display(out_cal)
render_calendar()

Output()

---
# 📋 Section H — Data Explorer Table

In [27]:
# ── Raw data explorer with search ─────────────────────────────────────────────
w_tbl_city = widgets.Combobox(
    options=TOP_CITIES,
    value=TOP_CITIES[0],
    description='🏙️ City:',
    style={'description_width': '55px'},
    layout=Layout(width='320px')
)
w_tbl_rows = widgets.IntSlider(
    value=20, min=5, max=100, step=5,
    description='Rows:',
    layout=Layout(width='320px')
)
out_tbl = Output()

DISPLAY_COLS = ['date','confirmed','daily_new','growth_rate','deaths','recovered','active','case_rate','country']

def render_table(*_):
    with out_tbl:
        clear_output(wait=True)
        cdf = city_data(w_tbl_city.value)
        show_cols = [c for c in DISPLAY_COLS if c in cdf.columns]
        tbl = cdf[show_cols].tail(w_tbl_rows.value).sort_values('date', ascending=False)
        tbl['date'] = tbl['date'].dt.strftime('%Y-%m-%d')
        for c in ['confirmed','daily_new','deaths','recovered','active']:
            if c in tbl:
                tbl[c] = tbl[c].apply(lambda x: f'{int(x):,}' if pd.notna(x) else '-')
        for c in ['growth_rate','case_rate']:
            if c in tbl:
                tbl[c] = tbl[c].apply(lambda x: f'{x:.2f}' if pd.notna(x) else '-')
        display(tbl.style
                   .set_table_styles([{
                       'selector': 'th',
                       'props': [('background-color','#4361ee'),('color','white'),
                                 ('font-weight','bold'),('font-size','13px'),('padding','8px')]
                   },{
                       'selector': 'td',
                       'props': [('font-size','12px'),('padding','6px')]
                   }])
                   .hide(axis='index')
        )

for w in [w_tbl_city, w_tbl_rows]:
    w.observe(render_table, names='value')

display(HTML('<hr><h2 style="font-family:Arial">📋 Data Explorer</h2>'))
display(HBox([w_tbl_city, w_tbl_rows]))
display(out_tbl)
render_table()

Output()